# Python + Pandas Data Exploration and Cleaning

In [1]:
import pandas as pd
import os
import csv
from pathlib import Path

root = Path.cwd()
csv_files = [str(p) for p in root.rglob('*.csv') if p.is_file()]
csv_files

detected_file = None
candidates = [p for p in csv_files if 'superstore' in os.path.basename(p).lower()]
for file_path in candidates:
    if 'cleaned' not in os.path.basename(file_path).lower():
        detected_file = file_path
        break
if detected_file is None and candidates:
    detected_file = candidates[0]
if detected_file is None:
    detected_file = csv_files[0]

print('Detected file:', detected_file)

for encoding in ['utf-8', 'utf-8-sig', 'latin1']:
    try:
        with open(detected_file, 'r', encoding=encoding) as handle:
            handle.read(2000)
        break
    except Exception:
        encoding = None

with open(detected_file, 'r', encoding='utf-8', errors='replace') as handle:
    sample = handle.read(4096)
    try:
        delimiter = csv.Sniffer().sniff(sample).delimiter
    except Exception:
        delimiter = ','

print('Detected encoding:', encoding)
print('Detected delimiter:', delimiter)

df = pd.read_csv(detected_file, encoding=encoding, sep=delimiter)

print('First 5 rows')
print(df.head())
print('Last 5 rows')
print(df.tail())
print('Columns')
print(df.columns.tolist())
print('Shape', df.shape)
print('Data types')
print(df.dtypes)
print('Info')
print(df.info())
print('Describe')
print(df.describe())

print('Missing values before cleaning')
print(df.isna().sum())

Detected file: c:\Users\ASUS\OneDrive\Desktop\Assignment_7\Superstore.csv
Detected encoding: utf-8
Detected delimiter: ,
First 5 rows
  Row ID        Order ID  Order Date   Ship Date       Ship Mode Customer ID  \
0      1  CA-2017-152156   11/8/2017  11/11/2017    Second Class    CG-12520   
1      2  CA-2017-152156   11/8/2017  11/11/2017    Second Class    CG-12520   
2      3  CA-2017-138688   6/12/2017   6/16/2017    Second Class    DV-13045   
3      4  US-2016-108966  10/11/2016  10/18/2016  Standard Class    SO-20335   
4      5  US-2016-108966  10/11/2016  10/18/2016  Standard Class    SO-20335   

     Customer Name    Segment        Country             City  ...  \
0      Claire Gute   Consumer  United States        Henderson  ...   
1      Claire Gute   Consumer  United States        Henderson  ...   
2  Darrin Van Huff  Corporate  United States      Los Angeles  ...   
3   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   
4   Sean O'Donnell   Consumer  Unit

In [2]:
# Basic cleaning
def find_matching_column(columns, names):
    for name in names:
        for col in columns:
            if name.lower() == col.lower():
                return col
    for name in names:
        for col in columns:
            if name.lower() in col.lower():
                return col
    return None

price_col = find_matching_column(df.columns, ['sales', 'price', 'unit price'])
quantity_col = find_matching_column(df.columns, ['quantity', 'qty'])

rows_before = len(df)
missing_before = int(df.isna().sum().sum())

if price_col is not None and quantity_col is not None:
    df['total_amount'] = df[price_col] * df[quantity_col]
else:
    df['total_amount'] = pd.Series([0] * len(df))

df = df.drop_duplicates()

for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna('Unknown')
    else:
        df[col] = df[col].fillna(0)

df = df.reset_index(drop=True)

rows_after = len(df)
missing_after = int(df.isna().sum().sum())
duplicates_removed = rows_before - rows_after

print('Rows before cleaning:', rows_before)
print('Rows after cleaning:', rows_after)
print('Missing values after cleaning')
print(df.isna().sum())
print('Duplicates removed:', duplicates_removed)
print('Derived column created:', 'total_amount' in df.columns)

Rows before cleaning: 10800
Rows after cleaning: 10296
Missing values after cleaning
Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
total_amount     0
dtype: int64
Duplicates removed: 504
Derived column created: True


In [3]:
# filter rows and select columns
filtered_df = df[df[quantity_col].astype(float) > 2].copy() if quantity_col else df.copy()
selected_columns = ['Customer ID', 'Customer Name', price_col, quantity_col, 'total_amount']
selected_columns = [c for c in selected_columns if c in df.columns]

filtered_df[selected_columns].head()

output_path = 'cleaned_superstore.csv'
df.to_csv(output_path, index=False)
print('Saved cleaned CSV to', output_path)


Saved cleaned CSV to cleaned_superstore.csv


In [4]:
summary = f"\nProject Summary\nRows before cleaning: {rows_before}\nRows after cleaning: {rows_after}\nMissing values handled: {missing_before - missing_after}\nDuplicates removed: {duplicates_removed}\nDerived column created: total_amount\n"
print(summary)


Project Summary
Rows before cleaning: 10800
Rows after cleaning: 10296
Missing values handled: 15325
Duplicates removed: 504
Derived column created: total_amount

